# ragkv on Kaggle / Colab

Runs the whole study in order: premise check (phase 1), main sweep (phase 2), plots.
Each long step writes to `results/` and skips work that is already there, so if the
session dies, just rerun the cell.

**Runtime:** GPU (T4 is enough for the 1B and 1.5B models). Kaggle: Settings -> Accelerator -> GPU,
and turn Internet on.

In [ ]:
# 1. get the code: upload ragkv.zip next to this notebook (Kaggle: add it as a dataset, then
#    !cp /kaggle/input/<dataset-name>/ragkv.zip . ), or clone your fork
!unzip -q -o ragkv.zip -d . 2>/dev/null || true
%cd ragkv
!pip install -q -e . datasets sentence-transformers
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
# 2. sanity check: the unit tests must pass on this machine before any number is trusted
!python -m pytest -q

## Models
Pick two. Llama needs a Hugging Face token with access to the gated repo; Qwen does not.

In [ ]:
MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "meta-llama/Llama-3.2-1B-Instruct"]
# from huggingface_hub import login; login()   # only needed for Llama
N = 300          # questions for the phase-1 check; use 500+ for the main sweep

In [ ]:
# 3. data: HotpotQA (distractor setting), scored two ways
!python scripts/prepare_data.py --source hotpot --n 500 --retriever bm25 --out data/hotpot_bm25.jsonl
!python scripts/prepare_data.py --source hotpot --n 500 --retriever BAAI/bge-reranker-base --out data/hotpot_bge.jsonl

## Phase 1: is the premise true?

Compares how well the retriever score, the question's attention, and key norm each
separate gold chunks from distractors (AUROC). Read it before starting the sweep:

* `retriever` clearly above `neg_knorm` (CIs not overlapping) -> the retriever prior carries information
  the attention-free signal lacks. Continue.
* `retriever` near `attn_mass` -> even better: you get attention-level guidance for free.
* `retriever` at or below `neg_knorm` -> stop. Either the retriever is weak (try the reranker data) or the
  idea does not hold on this model; that is a result too.
* `neg_knorm` near 0.5 -> key norm is a poor within-chunk scorer for this model. The sweep will
  show it; consider `retriever+snapkv` as the headline variant instead.

In [ ]:
for m in MODELS:
    tag = m.split("/")[-1]
    for data in ["hotpot_bm25", "hotpot_bge"]:
        print(f"\n===== {tag} on {data}")
        !python scripts/prior_analysis.py --model {m} --data data/{data}.jsonl --n {N} --out results/prior_{tag}_{data}.json

## Phase 2: main sweep

One JSONL per (model, retriever). `snapkv` needs eager attention, which uses more memory:
if it runs out of memory, drop it from the list or lower `--n`.

In [ ]:
POLICIES = "full global+knorm global+recency global+random uniform+knorm retriever+knorm oracle+knorm global+snapkv retriever+snapkv"
for m in MODELS:
    tag = m.split("/")[-1]
    for data in ["hotpot_bm25", "hotpot_bge"]:
        for seed in [0, 1, 2]:
            !python scripts/run_eval.py --model {m} --data data/{data}.jsonl --n 500 --seed {seed} \
                --policies {POLICIES} --ratios 0.1 0.2 0.3 0.5 --out results/{tag}_{data}.jsonl

In [ ]:
import glob, subprocess
for path in sorted(glob.glob("results/*_hotpot_*.jsonl")):
    out = path.replace(".jsonl", "")
    r = subprocess.run(["python", "scripts/make_plots.py", "--results", path, "--out", out], capture_output=True, text=True)
    print(path, "->", r.stdout.strip() or r.stderr.strip())


The three seeds shuffle the chunk order differently, so they are replicates of the same experiment; `make_plots.py` averages them per question before computing intervals. If you ever mix runs with different `--temperature`, `--floor` or `--order` in one file it will ask you to pick one with `--tag`.

In [ ]:
from IPython.display import Image, Markdown, display
for f in sorted(glob.glob("results/*_f1.png")):
    display(Image(f))
    display(Markdown(open(f.replace("_f1.png", "_summary.md")).read()))